# Inspect LongMemEval


In [5]:
import json

import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import display

DATASET_ID = "xiaowu0162/longmemeval-cleaned"
VARIANT = "oracle"
FILES = {
    "oracle": "longmemeval_oracle.json",
    "s": "longmemeval_s_cleaned.json",
}

revision = HfApi().dataset_info(DATASET_ID).sha
path = hf_hub_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    filename=FILES[VARIANT],
    revision=revision,
)
with open(path) as source:
    examples = json.load(source)

print(f"Loaded {len(examples)} examples ({VARIANT})")
print("Revision:", revision)
print("Fields:", list(examples[0]))

questions = pd.DataFrame(examples)
questions.index.name = "example_index"
questions["abstention"] = questions["question_id"].str.endswith("_abs")


Loaded 500 examples (oracle)
Revision: 98d7416c24c778c2fee6e6f3006e7a073259d48f
Fields: ['question_id', 'question_type', 'question', 'answer', 'question_date', 'haystack_dates', 'haystack_session_ids', 'haystack_sessions', 'answer_session_ids']


In [6]:
with pd.option_context("display.max_columns", None):
    display(questions.head())


,question_id,question_type,question,answer,question_date,haystack_dates,haystack_session_ids,haystack_sessions,answer_session_ids,abstention
example_index,,,,,,,,,,
0,gpt4_2655b836,temporal-reasoning,What was the first issue I had with my new car...,GPS system not functioning correctly,2023/04/10 (Mon) 23:07,"[2023/04/10 (Mon) 17:50, 2023/04/10 (Mon) 14:4...","[answer_4be1b6b4_2, answer_4be1b6b4_3, answer_...","[[{'role': 'user', 'content': 'I'm thinking of...","[answer_4be1b6b4_2, answer_4be1b6b4_3, answer_...",False
1,gpt4_2487a7cb,temporal-reasoning,"Which event did I attend first, the 'Effective...",'Data Analysis using Python' webinar,2023/05/28 (Sun) 06:47,"[2023/05/28 (Sun) 21:04, 2023/05/28 (Sun) 07:17]","[answer_1c6b85ea_1, answer_1c6b85ea_2]","[[{'role': 'user', 'content': 'I'm trying to g...","[answer_1c6b85ea_1, answer_1c6b85ea_2]",False
2,gpt4_76048e76,temporal-reasoning,Which vehicle did I take care of first in Febr...,bike,2023/03/10 (Fri) 23:15,"[2023/03/10 (Fri) 22:50, 2023/03/10 (Fri) 08:11]","[answer_b535969f_2, answer_b535969f_1]","[[{'role': 'user', 'content': 'I'm thinking of...","[answer_b535969f_2, answer_b535969f_1]",False
3,gpt4_2312f94c,temporal-reasoning,"Which device did I got first, the Samsung Gala...",Samsung Galaxy S22,2023/03/15 (Wed) 03:53,"[2023/03/15 (Wed) 00:56, 2023/03/15 (Wed) 10:31]","[answer_5328c3c2_1, answer_5328c3c2_2]","[[{'role': 'user', 'content': 'I'm planning a ...","[answer_5328c3c2_1, answer_5328c3c2_2]",False
4,0bb5a684,temporal-reasoning,How many days before the team meeting I was pr...,7 days. 8 days (including the last day) is als...,2023/01/13 (Fri) 19:39,"[2023/01/13 (Fri) 18:07, 2023/01/13 (Fri) 21:38]","[answer_e936197f_1, answer_e936197f_2]","[[{'role': 'user', 'content': 'I'm preparing f...","[answer_e936197f_1, answer_e936197f_2]",False


In [7]:
display(questions.groupby("question_type").size().rename("count").reset_index())


,question_type,count
0,knowledge-update,78
1,multi-session,133
2,single-session-assistant,56
3,single-session-preference,30
4,single-session-user,70
5,temporal-reasoning,133


In [8]:
EXAMPLE_INDEX = 0
example = examples[EXAMPLE_INDEX]

print("Question:", example["question"])
print("Answer:", example["answer"])


Question: What was the first issue I had with my new car after its first service?
Answer: GPS system not functioning correctly


In [9]:
SESSION_INDEX = 0
sessions = sorted(
    zip(example["haystack_dates"], example["haystack_session_ids"], example["haystack_sessions"]),
    key=lambda item: item[0],
)

if sessions:
    date, session_id, turns = sessions[SESSION_INDEX]
    print(f"Session {session_id} | {date}\n")
    for turn in turns:
        print(f"{turn['role']}: {turn['content']}\n")
else:
    print("This example has no supplied history sessions.")


Session answer_4be1b6b4_3 | 2023/04/10 (Mon) 14:47

user: I'm thinking of getting a car wax and detailing done soon. Can you give me some tips on what to look for when choosing a detailer?

assistant: Choosing the right detailer can make all the difference in getting your car looking its best. Here are some valuable tips to help you find a reliable and skilled detailer:

1. **Check online reviews**: Look up reviews on platforms like Google, Yelp, Facebook, and other review sites to see what previous customers have to say about their experience with the detailer. Pay attention to the overall rating and read the comments to get an idea of their strengths and weaknesses.
2. **Ask for referrals**: Ask friends, family, or coworkers who have had their cars detailed recently for recommendations. They can provide valuable insights and firsthand experience with the detailer.
3. **Check their credentials**: Look for certifications from reputable organizations like the International Detailing Ass